### WEC Performance Analysis - Phase 1 | Absolute Variant (Semi-Empirical)

Unified pipeline for the Absolute Performance Assessment of a 12-WEC fleet anchored on real wave telemetry (Waverider). This module consolidates model training, inference, anomaly flagging, visualization, and terminal reporting. It strictly enforces physical constraints by relying on pre-assigned dataset metadata (`Split_Role`) to prevent data leakage, seasonality bias, and synthetic imputation errors.

#### Pipeline stages
* **A** Data ingestion: Hard-drop of missing telemetry gaps (`IGNORE` tag). Zero generic data imputation.
* **B** Metadata-driven Train/Test split: Strict isolation of the "Golden Window" via `Split_Role`.
* **C** XGBoost regression training (physics-only features, temporal variables excluded) + metric extraction.
* **D** Full-dataset inference and absolute residual computation.
* **E** Absolute anomaly flagging via Statistical Process Control (residual < -3 * RMSE_train).
* **F** Visualization (3-panel, 2x2 GridSpec) with dynamic temporal cutoffs, saved to PNG.
* **G** Asset Performance Report emitted to the logger (comparing Baseline, Environmental False Positives, and True Faults).
* **H** Export of intermediate artefacts for Phase 2 (SFA) and Phase 3 (Decision Matrix).

In [23]:
from __future__ import annotations

import logging
import os
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from IPython.display import display

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import joblib
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# ---------------------------------------------------------------------------
# Logging configuration
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)
warnings.filterwarnings("ignore", category=UserWarning)


# ---------------------------------------------------------------------------
# Global constants
# ---------------------------------------------------------------------------

DATA_PATH: str = "dataset1/final_dataset/final_wec_fleet_2026.csv"
OUTPUT_PATH: str = "plots/phase1/d1_wec_phase1_absolute.png"

PHASE1_CSV_OUT: str = "dataset1/final_dataset/wec_phase1_outputs.csv"
PHASE1_MODEL_OUT: str = "dataset1/final_dataset/wec_phase1_xgboost.joblib"

TRAIN_CUTOFF_DATE: str = "2026-05-01"

# For dataset2
# DATA_PATH: str = "dataset2/wec_c5_mock_data_epochs.csv"
# OUTPUT_PATH: str = "plots/phase1/d2_wec_phase1_absolute.png"

# PHASE1_CSV_OUT: str = "dataset2/wec_phase1_outputs.csv"
# PHASE1_MODEL_OUT: str = "dataset2/wec_phase1_xgboost.joblib"

# TRAIN_CUTOFF_DATE: str = "2025-05-01"


# Temporal split fraction
TEST_FRACTION: float = 0.20


TIMESTAMP_COL: str = "PCTimeStamp"
TARGET_COL: str = "Energy_Generation_kW"
BUOY_ID_COL: str = "Buoy_ID"
EPOCH_COL: str = "Epoch_Marker"

# Expected buoy identifiers, ordered for consistent visualisation
BUOY_ORDER: List[str] = [f"Boia_{i}" for i in range(1, 13)]

# Model feature set
FEATURE_COLS: List[str] = [
    "Hs__m",
    "Te__s",
    "Wave_Power_Flux",
    "H1/3__m",
    "H1/10__m",
    "Hmax__m",
    "HTmax__m",
    "Havg__m",
    "Hsms__m",
    "NumberOfWaves",
    "THmax__s",
    "Tavg__s",
    "Tmax__s",
]


# Anomaly detection multiplier applied to RMSE_test
ANOMALY_SIGMA_FACTOR: float = 3

# Confidence band multiplier for the P10-P90 visual band
CONFIDENCE_Z: float = 1.28

# Physical generation bounds in kW
GEN_MIN_KW: float = 0.0
GEN_MAX_KW: float = 350.0

# XGBoost hyper-parameters
XGB_PARAMS: Dict = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}

# Definir os grupos de saude das boias para a analise particionada
HEALTHY_BUOYS: List[str] = [f"Boia_{i}" for i in range(1, 9)]
DEGRADED_BUOYS: List[str] = [f"Boia_{i}" for i in range(9, 13)]
ALL_BUOYS: List[str] = HEALTHY_BUOYS + DEGRADED_BUOYS

# Matplotlib / Seaborn aesthetics
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("paper", font_scale=1.2)

#### Core Functions
Definição das funções para engenharia de features, treino do XGBoost, inferência, visualização e geração de relatórios.

In [24]:
# ---------------------------------------------------------------------------
# Stage A -- Data ingestion and feature engineering
# ---------------------------------------------------------------------------
def load_and_engineer_features(csv_path: str) -> pd.DataFrame:
    """Load raw sensor data from CSV and discard unrecoverable telemetry gaps."""
    logger.info("Stage A -- Loading data from: %s", csv_path)
    df: pd.DataFrame = pd.read_csv(csv_path, parse_dates=[TIMESTAMP_COL])
    logger.info("Raw shape: %s", df.shape)

    df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)


    if "Split_Role" in df.columns:
        n_ignore = (df["Split_Role"] == "IGNORE").sum()
        df = df[df["Split_Role"] != "IGNORE"].copy()
        logger.info("Dropped %d rows marked as IGNORE (missing sensor data).", n_ignore)
    else:
        # Fallback caso o dataset nao tenha Split_Role
        df = df.dropna(subset=FEATURE_COLS).copy()

    logger.info("Preprocessed shape: %s", df.shape)
    return df

# ---------------------------------------------------------------------------
# Stage B -- Temporal split
# ---------------------------------------------------------------------------

def temporal_split(
    df: pd.DataFrame, 
    data_path: str = DATA_PATH,
    test_fraction: float = 0.20
) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series, pd.Series, pd.Series]:
    """
    Produce train/test splits based on the active dataset directory.
    dataset1 (Semi-empirical): Strict split using 'Split_Role' metadata.
    dataset2 (Legacy Synthetic): Chronological 80/20 split.
    """
    if "dataset1" in data_path:
        train_mask = df["Split_Role"] == "TRAIN"
        test_mask = df["Split_Role"].isin(["TEST_NOMINAL", "TEST_ANOMALY"])

        X_train = df.loc[train_mask, FEATURE_COLS].copy()
        y_train = df.loc[train_mask, TARGET_COL].copy()
        buoys_train = df.loc[train_mask, BUOY_ID_COL].copy()

        X_test = df.loc[test_mask, FEATURE_COLS].copy()
        y_test = df.loc[test_mask, TARGET_COL].copy()
        buoys_test = df.loc[test_mask, BUOY_ID_COL].copy()

        logger.info("Stage B -- Data split via Split_Role: train=%d rows | test=%d rows", len(X_train), len(X_test))
    elif "dataset2" in data_path:
        split_idx = int(len(df) * (1.0 - test_fraction))

        X = df[FEATURE_COLS].copy()
        y = df[TARGET_COL].copy()
        buoys = df[BUOY_ID_COL].copy()

        X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
        buoys_train, buoys_test = buoys.iloc[:split_idx], buoys.iloc[split_idx:]

        logger.info(
            "Stage B -- Legacy temporal split: train=%d rows | test=%d rows (%.0f%% / %.0f%%)",
            len(X_train), len(X_test), (1.0 - test_fraction) * 100, test_fraction * 100
        )
    else:
        raise ValueError(f"Unrecognized dataset path: {data_path}")
    

    return X_train, y_train, X_test, y_test, buoys_train, buoys_test


# ---------------------------------------------------------------------------
# Stage C -- Model training and metric extraction
# ---------------------------------------------------------------------------

def train_and_evaluate(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    buoys_test: pd.Series,
    params: Optional[Dict] = None,
) -> Tuple[XGBRegressor, float]:
    """
    Train an XGBoost regressor and return the fitted model together with the
    test-set RMSE extracted dynamically (never hardcoded).

    Parameters
    ----------
    X_train, y_train : training features and target.
    X_test, y_test   : held-out evaluation features and target.
    params           : optional XGBoost hyper-parameter dictionary.

    Returns
    -------
    model : XGBRegressor
        Fitted model ready for inference.
    rmse_test : float
        Root Mean Squared Error on the test partition.
    """
    effective_params: Dict = params or XGB_PARAMS
    model = XGBRegressor(**effective_params)

    logger.info(
        "Stage C -- Training XGBoost on %d samples, %d features...",
        len(X_train),
        X_train.shape[1],
    )
    model.fit(X_train, y_train)
    logger.info("Training complete.")

    # 1. Avaliar In-Sample (O Comportamento "Normal" Base)
    y_pred_train: np.ndarray = model.predict(X_train)
    rmse_train: float = float(np.sqrt(mean_squared_error(y_train, y_pred_train)))
    mae_train: float = float(mean_absolute_error(y_train, y_pred_train))
    r2_train: float = float(r2_score(y_train, y_pred_train))
    
    # 2. Avaliar Out- copiar este bloco integralmente, pois ele contém as tuof-Sample (Timeline: Ultimos 20% - Inclui anomalias da Epoch 3)
    y_pred_test: np.ndarray = model.predict(X_test)
    rmse_test_global: float = float(np.sqrt(mean_squared_error(y_test, y_pred_test)))
    mae_test_global: float = float(mean_absolute_error(y_test, y_pred_test))
    r2_test_global: float = float(r2_score(y_test, y_pred_test))

    # Estratificacao Out-of-Sample: Saudaveis (1-8) vs Degradadas (9-12)
    mask_healthy = buoys_test.isin(HEALTHY_BUOYS)
    mask_degraded = buoys_test.isin(DEGRADED_BUOYS)

    r2_test_healthy = r2_score(y_test[mask_healthy], y_pred_test[mask_healthy])
    rmse_test_healthy = np.sqrt(mean_squared_error(y_test[mask_healthy], y_pred_test[mask_healthy]))
    mae_test_healthy = float(mean_absolute_error(y_test[mask_healthy], y_pred_test[mask_healthy]))

    r2_test_degraded = r2_score(y_test[mask_degraded], y_pred_test[mask_degraded])
    rmse_test_degraded = np.sqrt(mean_squared_error(y_test[mask_degraded], y_pred_test[mask_degraded]))
    mae_test_degraded = float(mean_absolute_error(y_test[mask_degraded], y_pred_test[mask_degraded]))

    logger.info("--- Baseline Metrics (In-Sample / Epoch 1 / Train set) ---")
    logger.info("  RMSE : %.4f kW", rmse_train)
    logger.info("  MAE  : %.4f kW", mae_train)
    logger.info("  R^2  : %.4f ", r2_train)
    
    logger.info("--- Global Test Set Metrics (Out-of-Sample / Test set) ---")
    logger.info("  RMSE : %.4f kW ", rmse_test_global)
    logger.info("  MAE  : %.4f kW", mae_test_global)
    logger.info("  R^2  : %.4f ", r2_test_global)
    logger.info("-------------------------------------------------")

    logger.info("--- Healthy Test Set Metrics (Out-of-Sample / Test set) ---")
    logger.info("  RMSE : %.4f kW ", rmse_test_healthy)
    logger.info("  MAE  : %.4f kW", mae_test_healthy)
    logger.info("  R^2  : %.4f ", r2_test_healthy)
    logger.info("-------------------------------------------------")

    logger.info("--- Degraded Test Set Metrics (Out-of-Sample / Test set) ---")
    logger.info("  RMSE : %.4f kW  [used dynamically for anomalies]", rmse_test_degraded)
    logger.info("  MAE  : %.4f kW", mae_test_degraded)
    logger.info("  R^2  : %.4f ", r2_test_degraded)
    logger.info("-------------------------------------------------")

    importances: pd.Series = (
        pd.Series(model.feature_importances_, index=X_train.columns)
        .sort_values(ascending=False)
    )
    logger.info("Top-5 feature importances:\n%s", importances.head(5).to_string())

    # Usamos o RMSE do treino (o comportamento verdadeiramente normal) 
    # como a base da nossa banda de tolerância futura, que é muito mais rigoroso.
    return model, rmse_train



# ---------------------------------------------------------------------------
# Stage D & E -- Full-dataset inference and anomaly flagging
# ---------------------------------------------------------------------------
def run_inference_and_flag(df: pd.DataFrame, model: XGBRegressor, rmse_test: float) -> pd.DataFrame:
    """Apply the trained model to 100% of the dataset, compute residuals, and set the absolute anomaly flag."""
    logger.info("Stage D/E -- Full inference on %d rows + anomaly flagging", len(df))

    predictions: np.ndarray = model.predict(df[FEATURE_COLS])
    df = df.copy()
    df["Predicted_Energy_kW"] = np.clip(predictions, GEN_MIN_KW, GEN_MAX_KW)

    df["Absolute_Residual"] = df[TARGET_COL] - df["Predicted_Energy_kW"]

    anomaly_threshold: float = -ANOMALY_SIGMA_FACTOR * rmse_test
    df["Is_Absolute_Anomaly"] = df["Absolute_Residual"] < anomaly_threshold

    n_anomalies: int = int(df["Is_Absolute_Anomaly"].sum())
    logger.info("Global anomaly rate: %d / %d timestamps", n_anomalies, len(df))
    return df

# ---------------------------------------------------------------------------
# Stage F -- Visualisation
# ---------------------------------------------------------------------------
def _build_panel_1_feature_importance(ax: plt.Axes, model: XGBRegressor, top_n: int = 5) -> None:
    importances: pd.Series = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
    importances.tail(top_n).plot(kind="barh", color="#2c3e50", ax=ax)
    
    ax.set_title("A. Feature Importance\n(Phase 1 Output --> DEA Input)", fontweight="bold", fontsize=20)
    ax.set_xlabel("Importance Score (XGBoost)", fontsize=18)
    ax.tick_params(axis="both", labelsize=16)

# ---------------------------------------------------------------------------
# Stage F -- Visualisation
# ---------------------------------------------------------------------------
def _build_panel_3_forecast(
    ax: plt.Axes, 
    df: pd.DataFrame, 
    rmse_test: float, 
    buoy_id: str = "Boia_9",
    data_path: str = DATA_PATH,
    train_cutoff_legacy: Optional[str] = None
) -> None:
    half_band: float = CONFIDENCE_Z * rmse_test

    epoch3_start = df[df[EPOCH_COL] == 3][TIMESTAMP_COL].min()

    if "dataset1" in data_path:
        cutoff_dt = df[df["Split_Role"] == "TRAIN"][TIMESTAMP_COL].max()
        label_text = "Train Cutoff"
    elif "dataset2" in data_path:
        cutoff_str = train_cutoff_legacy if train_cutoff_legacy else "2025-05-18" # Fallback generico caso a constante falhe
        cutoff_dt = pd.to_datetime(cutoff_str)
        label_text = "Train Cutoff (80%)"
    else:
        raise ValueError(f"Unrecognized dataset path: {data_path}")

    df_buoy: pd.DataFrame = df[df[BUOY_ID_COL] == buoy_id].set_index(TIMESTAMP_COL)
    
    # Resample para visualização diária
    df_plot = df_buoy[[TARGET_COL, "Predicted_Energy_kW"]].resample("D").mean()
    df_plot["P10"] = (df_plot["Predicted_Energy_kW"] - half_band).clip(lower=GEN_MIN_KW)
    df_plot["P90"] = (df_plot["Predicted_Energy_kW"] + half_band).clip(upper=GEN_MAX_KW)

    train_mask = df_plot.index <= cutoff_dt
    df_train = df_plot[train_mask]
    df_test = df_plot[~train_mask]

    ax.plot(df_plot.index, df_plot[TARGET_COL], label="Actual Generation", color="#e74c3c", linewidth=2)
    ax.plot(df_train.index, df_train["Predicted_Energy_kW"], label="Model (Train / In-Sample)", color="gray", linestyle="--", linewidth=1.5)
    ax.plot(df_test.index, df_test["Predicted_Energy_kW"], label="Forecast (Test / Out-of-Sample)", color="#27ae60", linestyle="--", linewidth=2.5)
    ax.fill_between(df_test.index, df_test["P10"], df_test["P90"], color="#2ecc71", alpha=0.3, label=f"Confidence Band (+/-{CONFIDENCE_Z} RMSE = +/-{half_band:.1f} kW)")

    ax.axvline(cutoff_dt, color="black", linestyle="-", lw=1.5, label=label_text)
    ax.axvline(epoch3_start, color="gray", linestyle=":", alpha=0.7, label="Epoch 3 Start")

    ax.set_title(f"C. Probabilistic Forecast: {buoy_id} (Actual vs Expected)", fontweight="bold", fontsize=20)
    ax.set_ylabel("Power (kW)", fontsize=18)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.tick_params(axis="both", labelsize=16)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha="right", fontsize=16)
    ax.legend(fontsize=16, loc="lower left")


def _build_panel_2_anomaly_bar(ax: plt.Axes, df: pd.DataFrame, epoch: int = 3) -> None:
    df_epoch: pd.DataFrame = df[df[EPOCH_COL] == epoch].copy()
    anomaly_pct: pd.Series = df_epoch.groupby(BUOY_ID_COL)["Is_Absolute_Anomaly"].mean().reindex(BUOY_ORDER).fillna(0.0) * 100.0

    anomaly_pct.index = anomaly_pct.index.str.replace("Boia_", "Buoy ")

    colors: List[str] = ["#c0392b" if buoy in [f"Buoy {i}" for i in range(9, 13)] else "#2c3e50" for buoy in anomaly_pct.index]
    bars = ax.bar(anomaly_pct.index, anomaly_pct.values, color=colors, edgecolor="white", linewidth=0.6)

    ax.axhline(10.0, color="orange", linestyle="--", linewidth=1.2, label="10% Reference Line")

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="#2c3e50", label="Healthy Fleet (Buoys 1-8)"), 
        Patch(facecolor="#c0392b", label="Degraded Fleet (Buoys 9-12)")
    ]
    ax.legend(handles=legend_elements, fontsize=16, loc="upper left")
    
    ax.set_title(f"B. Absolute Anomaly Rate per Buoy - Epoch {epoch}\n(% Timestamps with Residual < -3 * RMSE_train)", fontweight="bold", fontsize=20)
    ax.set_xlabel("WEC Asset (Buoy ID)", fontsize=18)
    ax.set_ylabel("Anomaly Rate (%)", fontsize=18)
    ax.tick_params(axis="x", labelrotation=30, labelsize=16)
    ax.tick_params(axis="y", labelsize=16)
    ax.set_ylim(bottom=0)

def generate_figure(df: pd.DataFrame, model: XGBRegressor, rmse_test: float, output_path: str = OUTPUT_PATH) -> None:
    logger.info("Stage F -- Composing 3-panel figure")
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    fig = plt.figure(figsize=(18, 14))
    
    gs = fig.add_gridspec(nrows=2, ncols=2, hspace=0.45, wspace=0.22)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, :])


    _build_panel_1_feature_importance(ax1, model)
    _build_panel_2_anomaly_bar(ax2, df, epoch=3)
    _build_panel_3_forecast(ax3, df, rmse_test)

    plt.savefig(output_path, dpi=600, bbox_inches="tight")
    logger.info("Figure saved to: %s", Path(output_path).resolve())
    plt.close(fig)

# ---------------------------------------------------------------------------
# Stage G & H -- Reporting and Export\
# ---------------------------------------------------------------------------
def emit_asset_performance_report(df: pd.DataFrame, epoch: int = 3) -> None:
    df_epoch: pd.DataFrame = df[df[EPOCH_COL] == epoch].copy()
    separator: str = "=" * 70

    logger.info(separator)
    logger.info("ASSET PERFORMANCE REPORT -- Epoch %d", epoch)
    logger.info("Anomaly definition: Residual < -%.1f * RMSE_test", ANOMALY_SIGMA_FACTOR)
    logger.info(separator)

    mask_healthy = df_epoch[BUOY_ID_COL].isin(HEALTHY_BUOYS)
    mask_degraded = df_epoch[BUOY_ID_COL].isin(DEGRADED_BUOYS)
    
    y_true_h = df_epoch[mask_healthy][TARGET_COL]
    y_pred_h = df_epoch[mask_healthy]["Predicted_Energy_kW"]
    r2_healthy = r2_score(y_true_h, y_pred_h) if not y_true_h.empty else np.nan
    
    y_true_d = df_epoch[mask_degraded][TARGET_COL]
    y_pred_d = df_epoch[mask_degraded]["Predicted_Energy_kW"]
    r2_degraded = r2_score(y_true_d, y_pred_d) if not y_true_d.empty else np.nan
    
    logger.info("  Healthy Fleet R^2  : %.4f (Model retains accuracy)", r2_healthy)
    logger.info("  Degraded Fleet R^2 : %.4f (Metric collapse confirms anomaly)", r2_degraded)
    logger.info(separator)

    anomaly_rates: Dict[str, float] = {}
    for buoy in BUOY_ORDER:
        df_buoy: pd.DataFrame = df_epoch[df_epoch[BUOY_ID_COL] == buoy]
        if df_buoy.empty:
            continue
        pct: float = 100.0 * int(df_buoy["Is_Absolute_Anomaly"].sum()) / len(df_buoy)
        anomaly_rates[buoy] = pct
        logger.info("%s underperformed in %.2f%% of the timestamps.", buoy.replace("_", " "), pct)

    if anomaly_rates:
        worst_buoy: str = max(anomaly_rates, key=lambda b: anomaly_rates[b])
        logger.info(separator)
        logger.info("CONCLUSION -- Worst performing asset: %s (anomaly rate = %.2f%%).", worst_buoy.replace("_", " "), anomaly_rates[worst_buoy])

def export_artefacts_phase1(df: pd.DataFrame, model: XGBRegressor, rmse_test: float) -> None:
    logger.info("Stage H -- Exporting intermediate artefacts")
    
    export_cols = [TIMESTAMP_COL, BUOY_ID_COL, "Predicted_Energy_kW", "Absolute_Residual", "Is_Absolute_Anomaly"]
    
    for col in ["Split_Role", EPOCH_COL, "Wave_Power_Flux"]:
        if col in df.columns and col not in export_cols:
            export_cols.append(col)
            
    df_export = df[export_cols].copy()
    df_export["RMSE_test_dynamic"] = rmse_test
    
    os.makedirs(os.path.dirname(PHASE1_CSV_OUT), exist_ok=True)
    df_export.to_csv(PHASE1_CSV_OUT, index=False)
    joblib.dump(model, PHASE1_MODEL_OUT)
    
    logger.info("Export complete.")
    logger.info("=" * 60)

def export_artefacts_phase1(df: pd.DataFrame, model: XGBRegressor, rmse_test: float) -> None:
    logger.info("Stage H -- Exporting intermediate artefacts")


    export_cols = [TIMESTAMP_COL, BUOY_ID_COL, "Predicted_Energy_kW", "Absolute_Residual", "Is_Absolute_Anomaly"]
    df_export = df[export_cols].copy()
    df_export["RMSE_test_dynamic"] = rmse_test

    os.makedirs(os.path.dirname(PHASE1_CSV_OUT), exist_ok=True)
    df_export.to_csv(PHASE1_CSV_OUT, index=False)
    joblib.dump(model, PHASE1_MODEL_OUT)

    logger.info("Export complete.")
    logger.info("=" * 60)




#### Execution Block
Chamada sequencial das funções para correr a Fase 1 na totalidade (equivalente ao antigo bloco `__main__`).

In [25]:
logger.info("=" * 60)
logger.info("WEC Phase 1 -- Absolute Performance Analysis Execution")
logger.info("=" * 60)

# Stage A
df = load_and_engineer_features(DATA_PATH)

# Stage B
X_train, y_train, X_test, y_test, buoys_train, buoys_test = temporal_split(df)

# Stage C
model, rmse_test = train_and_evaluate(X_train, y_train, X_test, y_test, buoys_test)

# Stages D + E
df = run_inference_and_flag(df, model, rmse_test)

# Stage F
generate_figure(df, model, rmse_test)
X_train, y_train, X_test, y_test, buoys_train, buoys_test = temporal_split(df)
# Stage G
emit_asset_performance_report(df, epoch=1)
emit_asset_performance_report(df, epoch=2)
emit_asset_performance_report(df, epoch=3)

# Stage H 
export_artefacts_phase1(df, model, rmse_test)

logger.info("=" * 60)
logger.info("Pipeline completed successfully.")
logger.info("=" * 60)


2026-07-25 22:46:36 | INFO | ============================================================
2026-07-25 22:46:36 | INFO | WEC Phase 1 -- Absolute Performance Analysis Execution
2026-07-25 22:46:36 | INFO | ============================================================
2026-07-25 22:46:36 | INFO | Stage A -- Loading data from: dataset1/final_dataset/final_wec_fleet_2026.csv
2026-07-25 22:46:36 | INFO | Raw shape: (70272, 22)
2026-07-25 22:46:36 | INFO | Dropped 4152 rows marked as IGNORE (missing sensor data).
2026-07-25 22:46:36 | INFO | Preprocessed shape: (66120, 22)
2026-07-25 22:46:36 | INFO | Stage B -- Data split via Split_Role: train=18211 rows | test=47909 rows
2026-07-25 22:46:36 | INFO | Stage C -- Training XGBoost on 18211 samples, 13 features...
2026-07-25 22:46:38 | INFO | Training complete.
2026-07-25 22:46:38 | INFO | --- Baseline Metrics (In-Sample / Epoch 1 / Train set) ---
2026-07-25 22:46:38 | INFO |   RMSE : 5.7333 kW
2026-07-25 22:46:38 | INFO |   MAE  : 4.4939 kW
2026-

### WEC Performance Analysis - Phase 2 | Stochastic Frontier Analysis (SFA)

Stochastic Frontier Analysis for technical efficiency scoring and relative mechanical degradation detection across a 12-WEC fleet operating in a semi-empirical ocean environment.

#### Motivation
Phase 1 (XGBoost regression) identifies the absolute deviation of each WEC from its theoretical maximum. However, empirical testing demonstrates that absolute models fail to isolate mechanical faults from shared environmental anomalies (e.g., spectral spreading in Epoch 2) and naturally punish WECs located deeper in the wave farm due to spatial attenuation (Wake Effects).

SFA addresses this by dynamically estimating an empirical production frontier and decomposing the composite residual into two statistically distinct components:
`epsilon_i = v_i - u_i`

where:
* `v_i ~ N(0, sigma_v^2)` symmetric, shared noise (e.g., environmental shifts, sensor variance)
* `u_i ~ |N(0, sigma_u^2)|` one-sided, isolated technical inefficiency (e.g., PTO degradation)

#### Pipeline
1. `load_and_prepare` -- ingest CSV, compute WPF, apply log transform.
2. `fit_sfa_epoch1` -- MLE exclusively on the Epoch 1 "Golden Window".
3. `score_efficiency` -- Battese-Coelli estimator applied out-of-sample.
4. `compute_generation_deficit` -- back-transform frontier, extract true kW deficit.
5. `aggregate_rolling` -- 7-day rolling mean per buoy for visual stability.
6. `plot_timeseries` -- efficiency curves demonstrating environmental vs. mechanical drops.
7. `plot_residual_decomp` -- KDE statistical separation of v and u for Epoch 3.
8. `plot_triple_frontier` -- 3-panel frontier scatter generated iteratively for Epochs 1, 2, and 3.
9. `print_degradation_report` -- Terminal report with empirical deficit ranking.

In [26]:
from __future__ import annotations

import logging
import os
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib
# matplotlib.use("Agg") # Comentado para permitir que os graficos aparecam no Jupyter Notebook
%matplotlib inline 
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import minimize, OptimizeResult
from scipy.stats import norm

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("Phase2_SFA")
warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

TIMESTAMP_COL: str = "PCTimeStamp"
BUOY_COL: str = "Buoy_ID"
TARGET_COL: str = "Energy_Generation_kW"
WPF_COL: str = "Wave_Power_Flux"
EPOCH_COL: str = "Epoch_Marker"

GEN_MAX_KW: float = 350.0       # physical rated capacity of each WEC [kW]
LOG_EPS: float = 1e-6           # guard constant added before log
ROLLING_WINDOW: str = "7D"      # smoothing window for efficiency time series


PLOT_DIR: Path = Path("plots/phase2_SFA/d1")
PHASE2_CSV_OUT: str = "dataset1/final_dataset/wec_phase2_outputs.csv"

# PLOT_DIR: Path = Path("plots/phase2_SFA/d2")
# PHASE2_CSV_OUT: str = "dataset2/wec_phase2_outputs.csv"

# Colour palette used consistently across all plots
COLOR_HEALTHY: str = "#2471A3"
COLOR_DEGRADED: str = "#C0392B"
COLOR_FRONTIER: str = "#1A5276"

#### Core Functions
Definição das funções para modelação SFA, Maximum Likelihood Estimation (MLE), scoring de eficiência de Battese-Coelli e visualizações.

In [27]:

# ===========================================================================
# Section 1 -- Data Loading and Preparation
# ===========================================================================
def load_and_prepare(csv_path: str) -> pd.DataFrame:
    logger.info("Loading data from: %s", csv_path)
    df: pd.DataFrame = pd.read_csv(csv_path, parse_dates=[TIMESTAMP_COL])
    logger.info("Raw shape: %s", df.shape)

    if WPF_COL not in df.columns:
        logger.info("Column '%s' not found -- computing from Hs and Te", WPF_COL)
        df[WPF_COL] = 0.49 * df["Hs__m"] ** 2 * df["Te__s"]

    df[TARGET_COL] = df[TARGET_COL].clip(upper=GEN_MAX_KW)

    n_before: int = len(df)
    df = df[(df[WPF_COL] > 0) & (df[TARGET_COL] > 0)].copy()
    n_dropped: int = n_before - len(df)
    if n_dropped > 0:
        logger.warning("Dropped %d rows with non-positive WPF or output", n_dropped)

    for col in [WPF_COL, TARGET_COL]:
        n_nan: int = int(df[col].isna().sum())
        if n_nan > 0:
            df[col] = df[col].fillna(df[col].median())
            logger.info("Imputed %d NaN values in column '%s'", n_nan, col)

    df["ln_wpf"] = np.log(df[WPF_COL] + LOG_EPS)
    df["ln_y"] = np.log(df[TARGET_COL] + LOG_EPS)

    df = df.sort_values([TIMESTAMP_COL, BUOY_COL]).reset_index(drop=True)
    logger.info("Prepared shape: %s | Epochs present: %s", df.shape, sorted(df[EPOCH_COL].unique()))
    return df

# ===========================================================================
# Section 2 -- MLE Fitting on Epoch 1
# ===========================================================================
def _neg_log_likelihood(params: np.ndarray, ln_x: np.ndarray, ln_y: np.ndarray) -> float:
    beta0: float = params[0]
    beta1: float = params[1]
    sigma2: float = np.exp(params[2])
    lam: float = np.exp(params[3])

    sigma: float = np.sqrt(sigma2)
    epsilon: np.ndarray = ln_y - beta0 - beta1 * ln_x
    n: int = len(epsilon)

    z: np.ndarray = -lam * epsilon / sigma
    log_phi: np.ndarray = norm.logcdf(z)

    log_lik: float = (
        n * np.log(2)
        - n * np.log(sigma)
        - n * 0.5 * np.log(2.0 * np.pi)
        + log_phi.sum()
        - (epsilon ** 2).sum() / (2.0 * sigma2)
    )
    return -log_lik

def fit_sfa_epoch1(df: pd.DataFrame) -> Dict:
    df_e1: pd.DataFrame = df[(df[EPOCH_COL] == 1) & (df[TARGET_COL] < 345.0)].copy()
    ln_x: np.ndarray = df_e1["ln_wpf"].values
    ln_y: np.ndarray = df_e1["ln_y"].values

    logger.info("Fitting SFA on Epoch 1 Ramp-up Region: %d observations from %d buoys", len(ln_x), df_e1[BUOY_COL].nunique())

    initial_points: List[List[float]] = [
        [3.0, 0.8, np.log(0.5), np.log(1.0)],
        [2.5, 0.9, np.log(0.2), np.log(2.0)],
        [3.5, 0.7, np.log(1.0), np.log(0.5)],
        [3.0, 1.0, np.log(0.1), np.log(3.0)],
    ]

    best_result: Optional[OptimizeResult] = None
    best_nll: float = np.inf

    for x0 in initial_points:
        try:
            res: OptimizeResult = minimize(
                _neg_log_likelihood,
                x0=np.array(x0),
                args=(ln_x, ln_y),
                method="L-BFGS-B",
                options={"maxiter": 5000, "ftol": 1e-12, "gtol": 1e-8},
            )
            if res.fun < best_nll:
                best_nll = res.fun
                best_result = res
        except Exception as exc:
            logger.warning("Optimisation failed for starting point %s: %s", x0, exc)

    if best_result is None or not best_result.success:
        logger.warning("MLE did not converge cleanly -- check model or data quality")

    beta0: float = best_result.x[0]
    beta1: float = best_result.x[1]
    sigma2: float = np.exp(best_result.x[2])
    lam: float = np.exp(best_result.x[3])

    sigma_u2: float = sigma2 * lam ** 2 / (1.0 + lam ** 2)
    sigma_v2: float = sigma2 * 1.0 / (1.0 + lam ** 2)
    sigma_star2: float = sigma_u2 * sigma_v2 / sigma2

    params: Dict = {
        "beta0": beta0, "beta1": beta1, "sigma2": sigma2,
        "lambda_": lam, "sigma_u2": sigma_u2, "sigma_v2": sigma_v2,
        "sigma_star2": sigma_star2, "converged": best_result.success, "nll": best_nll,
    }

    logger.info("MLE results (Epoch 1 frontier):")
    logger.info("  beta_0    = %+.6f", beta0)
    logger.info("  beta_1    = %+.6f  (output elasticity)", beta1)
    logger.info("  lambda    = %.6f   (sigma_u / sigma_v)", lam)
    logger.info("  sigma^2   = %.6f   (total error variance)", sigma2)
    logger.info("  sigma_u^2 = %.6f   (inefficiency variance)", sigma_u2)
    logger.info("  sigma_v^2 = %.6f   (noise variance)", sigma_v2)
    logger.info("  converged = %s | NLL = %.4f", best_result.success, best_nll)

    return params


# ===========================================================================
# Section 3 -- Efficiency Scoring
# ===========================================================================
def score_efficiency(df: pd.DataFrame, params: Dict) -> pd.DataFrame:
    beta0: float = params["beta0"]
    beta1: float = params["beta1"]
    sigma2: float = params["sigma2"]
    sigma_u2: float = params["sigma_u2"]
    sigma_star2: float = params["sigma_star2"]
    sigma_star: float = np.sqrt(sigma_star2)

    epsilon: np.ndarray = df["ln_y"].values - beta0 - beta1 * df["ln_wpf"].values
    mu_star: np.ndarray = -epsilon * sigma_u2 / sigma2

    ratio: np.ndarray = mu_star / sigma_star
    te: np.ndarray = (
        np.exp(-mu_star + sigma_star2 / 2.0)
        * norm.cdf(ratio - sigma_star)
        / np.maximum(norm.cdf(ratio), 1e-15)
    )
    te = np.clip(te, 0.0, 1.0)

    df = df.copy()
    df["epsilon"] = epsilon
    df["mu_star"] = mu_star
    df["sigma_noise_hat"] = epsilon - (-mu_star)
    df["SFA_Efficiency"] = te

    logger.info(
        "Efficiency scoring complete | mean TE = %.4f | min = %.4f | max = %.4f",
        te.mean(), te.min(), te.max()
    )
    
    summary = (
        df.groupby([EPOCH_COL, BUOY_COL])["SFA_Efficiency"]
        .mean()
        .unstack(BUOY_COL)
        .round(4)
    )
    logger.info("Mean SFA efficiency per epoch and buoy:\n%s", summary.to_string())

    return df

# ===========================================================================
# Section 4 -- Generation Deficit
# ===========================================================================
def compute_generation_deficit(df: pd.DataFrame, params: Dict) -> pd.DataFrame:
    beta0: float = params["beta0"]
    beta1: float = params["beta1"]

    expected_y: np.ndarray = np.exp(beta0 + beta1 * np.log(df[WPF_COL].values + LOG_EPS))
    expected_y = np.clip(expected_y, 0.0, GEN_MAX_KW)

    df = df.copy()
    df["Expected_Y_kW"] = expected_y
    df["Generation_Deficit_kW"] = df["Expected_Y_kW"] - df[TARGET_COL]

    total_deficit_mwh: float = df["Generation_Deficit_kW"].clip(lower=0.0).sum() / 2000.0
    logger.info(
        "Generation deficit computed | mean = %.2f kW | total (positive) = %.1f MWh",
        df["Generation_Deficit_kW"].mean(), total_deficit_mwh
    )
    return df

# ===========================================================================
# Section 5 -- Rolling Aggregation
# ===========================================================================
def aggregate_rolling(df: pd.DataFrame) -> pd.DataFrame:
    df_pivot: pd.DataFrame = df.pivot_table(index=TIMESTAMP_COL, columns=BUOY_COL, values="SFA_Efficiency")
    df_pivot.columns.name = None
    rolling: pd.DataFrame = df_pivot.rolling(ROLLING_WINDOW, min_periods=1).mean()
    logger.info("7-day rolling means computed, shape: %s", rolling.shape)
    return rolling

# ===========================================================================
# Section 6 to 8 -- Visualisations
# ===========================================================================
def _epoch_boundaries(df: pd.DataFrame) -> Dict[int, pd.Timestamp]:
    return {int(epoch): df[df[EPOCH_COL] == epoch][TIMESTAMP_COL].min() for epoch in sorted(df[EPOCH_COL].unique())}

def plot_timeseries(rolling: pd.DataFrame, epoch_bounds: Dict[int, pd.Timestamp], save_path: str) -> None:
    fig, ax = plt.subplots(figsize=(16, 7))
    palette_healthy: List = sns.color_palette("Blues_r", n_colors=len(HEALTHY_BUOYS))
    palette_degraded: List = sns.color_palette("Reds_r", n_colors=len(DEGRADED_BUOYS))

    for i, buoy in enumerate(HEALTHY_BUOYS):
        if buoy in rolling.columns:
            ax.plot(rolling.index, rolling[buoy], color=palette_healthy[i], linewidth=1.4, alpha=0.85, label=buoy.replace("Boia_", "Buoy "))

    for i, buoy in enumerate(DEGRADED_BUOYS):
        if buoy in rolling.columns:
            ax.plot(rolling.index, rolling[buoy], color=palette_degraded[i], linewidth=2.2, alpha=0.95, label=f"{buoy.replace('Boia_', 'Buoy ')} (degraded)", linestyle="--")

    epoch_colors: Dict[int, str] = {1: "#555555", 2: "#E67E22", 3: "#C0392B"}
    epoch_labels: Dict[int, str] = {1: "Epoch 1\n(Golden Period)", 2: "Epoch 2\n(-15% global)", 3: "Epoch 3\n(PTO fault Buoys 9-12)"}
    
    for epoch, ts in epoch_bounds.items():
        ax.axvline(ts, color=epoch_colors[epoch], linestyle=":", linewidth=1.4, alpha=0.7)
        ax.text(ts, 0.04, epoch_labels[epoch], fontsize=14, color=epoch_colors[epoch], ha="left", va="bottom")

    ax.axhspan(0.0, 0.55, alpha=0.07, color="#C0392B", label="Severe degradation zone (<0.55)")
    ax.set_ylim(0.0, 1.08)
    ax.set_ylabel("SFA Technical Efficiency (TE)", fontsize=16)
    ax.set_xlabel("Date", fontsize=16)
    ax.tick_params(axis="y", labelsize=14)
    
    ax.set_title("WEC Phase 2 - SFA Technical Efficiency: 7-Day Rolling Mean\nEpoch 2: Spectral Spreading Penalty | Epoch 3: isolated PTO fault (Buoys 9-12)", fontsize=18, fontweight="bold")
    
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=25, ha="right", fontsize=14)
    ax.legend(fontsize=14, ncol=3, loc="upper right", framealpha=0.9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.close(fig)
    logger.info("Timeseries plot saved to: %s", save_path)

def plot_residual_decomposition(df: pd.DataFrame, save_path: str) -> None:
    df_e3: pd.DataFrame = df[df[EPOCH_COL] == 3].copy()
    df_e3["Group"] = df_e3[BUOY_COL].apply(lambda b: "Healthy (Buoys 1-8)" if b in HEALTHY_BUOYS else "Degraded (Buoys 9-12)")
    df_e3["u_hat"] = df_e3["mu_star"].clip(lower=0)
    df_e3["v_hat"] = df_e3["epsilon"] + df_e3["u_hat"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    palette_grp: Dict[str, str] = {"Healthy (Buoys 1-8)": COLOR_HEALTHY, "Degraded (Buoys 9-12)": COLOR_DEGRADED}

    ax_left = axes[0]
    for grp, sub in df_e3.groupby("Group"):
        sns.kdeplot(sub["epsilon"], ax=ax_left, label=grp, color=palette_grp[grp], linewidth=2.2, fill=True, alpha=0.25)
    ax_left.axvline(0, color="black", linestyle="--", linewidth=1.2, label="Zero residual")
    ax_left.set_xlabel("Composite Residual epsilon = ln(Y) - frontier", fontsize=16)
    ax_left.set_ylabel("Density", fontsize=16)
    ax_left.set_title("Composite Residual Distribution\nEpoch 3 (by operational group)", fontweight="bold", fontsize=18)
    ax_left.tick_params(axis="both", labelsize=14)
    ax_left.legend(fontsize=14)
    ax_left.grid(True, alpha=0.25)

    ax_right = axes[1]
    for buoy, color, label, col_name in [("Boia_1", COLOR_HEALTHY, "Buoy 1 - noise v (symmetric)", "v_hat"), ("Boia_9", COLOR_DEGRADED, "Buoy 9 - inferred inefficiency u", "u_hat")]:
        sub = df_e3[df_e3[BUOY_COL] == buoy]
        sns.kdeplot(sub[col_name], ax=ax_right, label=label, color=color, linewidth=2.2, fill=True, alpha=0.22)
    ax_right.axvline(0, color="black", linestyle="--", linewidth=1.2)
    ax_right.set_xlabel("Error component magnitude", fontsize=16)
    ax_right.set_ylabel("Density", fontsize=16)
    ax_right.set_title("SFA Error Decomposition: v vs u\nEpoch 3 (representative buoys)", fontweight="bold", fontsize=18)
    ax_right.tick_params(axis="both", labelsize=14)
    ax_right.legend(fontsize=14)
    ax_right.grid(True, alpha=0.25)

    fig.suptitle("SFA Residual Analysis - Epoch 3: Statistical Separation of Noise and Inefficiency", fontsize=20, fontweight="bold")

    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.close(fig)
    logger.info("Residual decomposition plot saved to: %s", save_path)

def _compute_frontier_curve(df_epoch: pd.DataFrame, params: Dict, n_points: int = 300) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    beta0: float = params["beta0"]
    beta1: float = params["beta1"]
    sigma_v: float = np.sqrt(params["sigma_v2"])

    wpf_range: np.ndarray = np.linspace(df_epoch[WPF_COL].quantile(0.01), df_epoch[WPF_COL].quantile(0.99), n_points)
    frontier_y: np.ndarray = np.clip(np.exp(beta0 + beta1 * np.log(wpf_range + LOG_EPS)), 0.0, GEN_MAX_KW)
    upper: np.ndarray = np.clip(frontier_y * np.exp(+sigma_v), 0.0, GEN_MAX_KW)
    lower: np.ndarray = np.clip(frontier_y * np.exp(-sigma_v), 0.0, GEN_MAX_KW)
    return wpf_range, frontier_y, lower, upper

def _draw_frontier_overlay(ax: plt.Axes, wpf_range: np.ndarray, frontier_y: np.ndarray, lower: np.ndarray, upper: np.ndarray, show_legend_label: bool = True) -> None:
    lbl_frontier = r"SFA Deterministic Frontier: $\hat{Y}=\exp(\hat\beta_0+\hat\beta_1\ln WPF)$" if show_legend_label else "_nolegend_"
    lbl_band = r"$\pm1\,\sigma_v$ stochastic band (noise scatter)" if show_legend_label else "_nolegend_"
    ax.plot(wpf_range, frontier_y, color=COLOR_FRONTIER, linewidth=2.2, linestyle="-", label=lbl_frontier, zorder=5)
    ax.fill_between(wpf_range, lower, upper, alpha=0.12, color=COLOR_HEALTHY, label=lbl_band)


def _buoy_color(buoy: str) -> str:
    return COLOR_DEGRADED if buoy in DEGRADED_BUOYS else COLOR_HEALTHY


def plot_triple_frontier(df: pd.DataFrame, params: Dict, save_path: str, epoch: int = 3, scatter_sample_per_buoy: int = 120) -> None:
    logger.info("Building triple frontier scatter for Epoch %d", epoch)
    df_epoch: pd.DataFrame = df[df[EPOCH_COL] == epoch].copy()
    wpf_range, frontier_y, lower, upper = _compute_frontier_curve(df_epoch, params)

    epoch_start: pd.Timestamp = df_epoch[TIMESTAMP_COL].min()
    epoch_end: pd.Timestamp = df_epoch[TIMESTAMP_COL].max()
    epoch_mid: pd.Timestamp = epoch_start + (epoch_end - epoch_start) / 2

    ts_per_buoy: List[set] = [set(df_epoch[df_epoch[BUOY_COL] == b][TIMESTAMP_COL].tolist()) for b in ALL_BUOYS if b in df_epoch[BUOY_COL].unique()]
    
    if ts_per_buoy:
        common_ts: set = ts_per_buoy[0].intersection(*ts_per_buoy[1:])
    else:
        common_ts = set()

    snapshot_ts: Optional[pd.Timestamp] = None
    if common_ts:
        common_series: pd.Series = pd.Series(sorted(common_ts))
        snapshot_ts = common_series.iloc[(common_series - epoch_mid).abs().argmin()]
        logger.info("Snapshot timestamp (Panel B): %s", snapshot_ts)
    else:
        logger.warning("No common timestamp found for all buoys in Epoch %d - Panel B will be empty", epoch)
    
    df_mean: pd.DataFrame = df_epoch.groupby(BUOY_COL)[[WPF_COL, TARGET_COL, "Expected_Y_kW", "Generation_Deficit_kW", "SFA_Efficiency"]].mean().reindex(ALL_BUOYS).dropna()

    fig, axes = plt.subplots(1, 3, figsize=(20, 9.5), sharex=True, sharey=True)
    panel_titles: List[str] = [
        f"A - Population View\n(sampled 30-min observations, Epoch {epoch})",
        f"B - Instantaneous Snapshot\n(single common timestamp: {snapshot_ts.strftime('%Y-%m-%d %H:%M') if snapshot_ts else 'N/A'})",
        f"C - Mean Operating Point\n(per-buoy epoch mean)"
    ]

    # Panel A
    ax_a: plt.Axes = axes[0]
    sampled_chunks = []
    for _, group_data in df_epoch.groupby(BUOY_COL):
        n_samples = min(len(group_data), scatter_sample_per_buoy)
        sampled_chunks.append(group_data.sample(n=n_samples, random_state=42))
    df_sample_a: pd.DataFrame = pd.concat(sampled_chunks, ignore_index=True)

    for buoy in ALL_BUOYS:
        sub = df_sample_a[df_sample_a[BUOY_COL] == buoy]
        if not sub.empty:
            buoy_label = buoy.replace("Boia_", "Buoy ")
            ax_a.scatter(sub[WPF_COL], sub[TARGET_COL], color=_buoy_color(buoy), alpha=0.25, s=14, edgecolors="none", label=buoy_label)
    _draw_frontier_overlay(ax_a, wpf_range, frontier_y, lower, upper, show_legend_label=True)
    ax_a.set_title(panel_titles[0], fontsize=18, fontweight="bold")
    ax_a.set_xlabel("Wave Power Flux (WPF) [kW/m]", fontsize=18)
    ax_a.set_ylabel("Energy Generation [kW]", fontsize=18)
    ax_a.tick_params(labelsize=16)
    ax_a.grid(True, alpha=0.2)

    # Panel B
    ax_b: plt.Axes = axes[1]
    _draw_frontier_overlay(ax_b, wpf_range, frontier_y, lower, upper, show_legend_label=False)
    if snapshot_ts is not None:
        df_snap: pd.DataFrame = df_epoch[df_epoch[TIMESTAMP_COL] == snapshot_ts]
        for buoy in ALL_BUOYS:
            row = df_snap[df_snap[BUOY_COL] == buoy]
            if not row.empty:
                x_val, y_val = row[WPF_COL].values[0], row[TARGET_COL].values[0]
                y_exp, deficit = row["Expected_Y_kW"].values[0], row["Generation_Deficit_kW"].values[0]
                buoy_label = buoy.replace("Boia_", "Buoy ")
                
                if buoy in DEGRADED_BUOYS:
                    ax_b.vlines(x=x_val, ymin=y_val, ymax=y_exp, color=COLOR_DEGRADED, linestyle='--', linewidth=1.5, zorder=5)
                    ax_b.text(x_val + 0.5, (y_val + y_exp) / 2, f"-{deficit:.0f} kW", color=COLOR_DEGRADED, fontsize=15, fontweight="bold", va='center')
                ax_b.scatter(x_val, y_val, color=_buoy_color(buoy), s=120, alpha=0.92, edgecolors="white", linewidths=0.8, zorder=6, label=buoy_label)
                buoy_idx: str = buoy.split("_")[-1]
                ax_b.annotate(buoy_idx, xy=(x_val, y_val), xytext=(3, 4), textcoords="offset points", fontsize=14, color=_buoy_color(buoy), fontweight="bold")
    ax_b.set_title(panel_titles[1], fontsize=18, fontweight="bold")
    ax_b.set_xlabel("Wave Power Flux (WPF) [kW/m]", fontsize=18)
    ax_b.tick_params(labelsize=16)
    ax_b.grid(True, alpha=0.2)

    # Panel C
    ax_c: plt.Axes = axes[2]
    _draw_frontier_overlay(ax_c, wpf_range, frontier_y, lower, upper, show_legend_label=False)
    best_buoy, worst_buoy = df_mean["SFA_Efficiency"].idxmax(), df_mean["SFA_Efficiency"].idxmin()
    for buoy in df_mean.index:
        row_wpf, row_y = df_mean.loc[buoy, WPF_COL], df_mean.loc[buoy, TARGET_COL]
        buoy_label = buoy.replace("Boia_", "Buoy ")
        ax_c.scatter(row_wpf, row_y, marker="X", color=_buoy_color(buoy), s=160, alpha=0.95, edgecolors="white", linewidths=0.8, zorder=6, label=buoy_label)
        
        if buoy == worst_buoy:
            deficit, te = df_mean.loc[buoy, "Generation_Deficit_kW"], df_mean.loc[buoy, "SFA_Efficiency"]
            bbox_props = dict(boxstyle="round,pad=0.3", fc="white", ec=COLOR_DEGRADED, lw=1.2, alpha=0.9)
            # Corrigido o \\n para \n e alterado a label para usar Buoy
            ax_c.annotate(f"Worst Asset ({buoy_label})\nTE: {te:.2f} | Deficit: -{deficit:.1f} kW", xy=(row_wpf, row_y), xytext=(row_wpf + 2, row_y - 50), arrowprops=dict(facecolor=COLOR_DEGRADED, shrink=0.05, width=1.5, headwidth=5), fontsize=13, fontweight="bold", color=COLOR_DEGRADED, bbox=bbox_props, zorder=10)
        elif buoy == best_buoy:
            te = df_mean.loc[buoy, "SFA_Efficiency"]
            bbox_props = dict(boxstyle="round,pad=0.3", fc="white", ec=COLOR_HEALTHY, lw=1.2, alpha=0.9)
            # Corrigido o \\n para \n e alterado a label para usar Buoy
            ax_c.annotate(f"Best Asset ({buoy_label})\nTE: {te:.2f}", xy=(row_wpf, row_y), xytext=(row_wpf - 18, row_y + 40), arrowprops=dict(facecolor=COLOR_HEALTHY, shrink=0.05, width=1.5, headwidth=5), fontsize=13, fontweight="bold", color=COLOR_HEALTHY, bbox=bbox_props, zorder=10)
        else:
            buoy_idx = buoy.split("_")[-1]
            ax_c.annotate(buoy_idx, xy=(row_wpf, row_y), xytext=(4, 4), textcoords="offset points", fontsize=14, color=_buoy_color(buoy), fontweight="bold")
    ax_c.set_title(panel_titles[2], fontsize=18, fontweight="bold")
    ax_c.set_xlabel("Wave Power Flux (WPF) [kW/m]", fontsize=18)
    ax_c.tick_params(labelsize=16)
    ax_c.grid(True, alpha=0.2)

    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    legend_elements = [
        Patch(facecolor=COLOR_HEALTHY,  label="Healthy fleet (Buoys 1-8)"),
        Patch(facecolor=COLOR_DEGRADED, label="Degraded fleet (Buoys 9-12)"),
        Line2D([0], [0], color=COLOR_FRONTIER, linewidth=2, label="SFA Deterministic Frontier"),
        Patch(facecolor=COLOR_HEALTHY, alpha=0.20, label=r"$\pm1\,\sigma_v$ Stochastic Band"),
    ]
    
    
    # Subtitulo redundante removido e atualizado
    fig.suptitle(f"WEC Phase 2 - SFA Triple Frontier Analysis (Epoch {epoch})", fontsize=18, fontweight="bold", y=1.01)

    fig.legend(handles=legend_elements, loc="upper center", ncol=4, fontsize=18, framealpha=0.9, bbox_to_anchor=(0.5, 0.01))
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.close(fig)
    logger.info("Triple frontier scatter saved to: %s", save_path)
    

# ===========================================================================
# Section 9 & 10 -- Reporting and Export
# ===========================================================================
def print_degradation_report(df: pd.DataFrame) -> None:
    separator: str = "=" * 72
    pivot: pd.DataFrame = (
        df.groupby([EPOCH_COL, BUOY_COL])["SFA_Efficiency"]
        .mean()
        .unstack(EPOCH_COL)
        .rename(columns={
            1: "Epoch1_base",
            2: "Epoch 2_Sub_optimal_Spectrum",
            3: "Epoch3_fault",
        })
    )
    pivot["Delta_E1_to_E3"] = pivot["Epoch3_fault"] - pivot["Epoch1_base"]
    pivot["Status"] = pivot["Delta_E1_to_E3"].apply(
        lambda d: "DEGRADATION DETECTED" if d < -0.20 else "normal"
    )

    logger.info("\n" + separator)
    logger.info("SFA DEGRADATION REPORT -- Phase 2 Summary")
    logger.info(separator)
    logger.info("Part 1 -- Cross-Epoch Efficiency Summary:")
    logger.info("\n" + pivot.round(4).to_string())
    logger.info(separator)

    df_e3: pd.DataFrame = df[df[EPOCH_COL] == 3].copy()
    deficit_summary: pd.DataFrame = (
        df_e3.groupby(BUOY_COL)
        .agg(
            Mean_SFA_Efficiency=("SFA_Efficiency", "mean"),
            Mean_Deficit_kW=("Generation_Deficit_kW", "mean"),
        )
        .reindex(ALL_BUOYS)
        .dropna()
        .sort_values("Mean_Deficit_kW", ascending=False)
        .round(4)
    )

    logger.info("Part 2 -- Epoch 3 Asset Criticality Ranking (descending deficit):")
    logger.info("\n" + deficit_summary.to_string())
    logger.info(separator)

    for buoy, row in deficit_summary.iterrows():
        logger.info(
            "%s | Mean SFA Efficiency = %.4f | Mean Generation Deficit = %.2f kW",
            str(buoy).replace("_", " "), row["Mean_SFA_Efficiency"], row["Mean_Deficit_kW"]
        )

    worst_buoy: str = str(deficit_summary.index[0])
    worst_deficit: float = float(deficit_summary.iloc[0]["Mean_Deficit_kW"])
    worst_efficiency: float = float(deficit_summary.iloc[0]["Mean_SFA_Efficiency"])

    logger.info(separator)
    logger.info("CONCLUSION -- Worst performing asset: %s", worst_buoy.replace("_", " "))
    logger.info("  Mean SFA Technical Efficiency : %.4f (%.1f%% of frontier)", worst_efficiency, worst_efficiency * 100.0)
    logger.info("  Mean Generation Deficit        : %.2f kW per observation", worst_deficit)
    logger.info("Recommendation: prioritise inspection and PTO maintenance of %s.", worst_buoy.replace("_", " "))
    logger.info(separator + "\n")

def export_artefacts(df: pd.DataFrame) -> None:
    logger.info("Exporting intermediate artefacts for Phase 3")
    export_cols = [TIMESTAMP_COL, BUOY_COL, EPOCH_COL, "SFA_Efficiency", "Generation_Deficit_kW"]
    df_export = df[export_cols].copy()
    
    os.makedirs(os.path.dirname(PHASE2_CSV_OUT), exist_ok=True)
    df_export.to_csv(PHASE2_CSV_OUT, index=False)
    logger.info("Phase 2 data contract exported to: %s", PHASE2_CSV_OUT)

#### Execution Block
Chamada sequencial das funções para correr a Fase 2. Inclui a geração de dados sintéticos caso o ficheiro CSV original não exista.

In [28]:
logger.info("=" * 72)
logger.info("WEC Phase 2 -- Stochastic Frontier Analysis (SFA)")
logger.info("=" * 72)

PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Stage 1 -- Load or Generate Mock Data
# ------------------------------------------------------------------
if os.path.exists(DATA_PATH):
    df: pd.DataFrame = load_and_prepare(DATA_PATH)
else:
    logger.warning("CSV not found at '%s' -- generating synthetic dataset", DATA_PATH)
    rng = np.random.default_rng(42)
    n_per_buoy: int = 2400
    records: List[pd.DataFrame] = []
    
    for buoy in ALL_BUOYS:
        for epoch in [1, 2, 3]:
            hs: np.ndarray = rng.uniform(0.8, 5.0, n_per_buoy)
            te: np.ndarray = rng.uniform(5.0, 15.0, n_per_buoy)
            wpf: np.ndarray = 0.49 * hs ** 2 * te
            base_eff: float = 1.0 if epoch == 1 else 0.85 if epoch == 2 else (0.45 if buoy in DEGRADED_BUOYS else 0.90)
            energy: np.ndarray = (base_eff * 2.8 * wpf + rng.normal(0, 12, n_per_buoy)).clip(1.0, GEN_MAX_KW)
            start: pd.Timestamp = pd.Timestamp(f"2025-0{2 + epoch}-01")
            ts: pd.DatetimeIndex = pd.date_range(start, periods=n_per_buoy, freq="30min")

            records.append(pd.DataFrame({
                TIMESTAMP_COL: ts,
                BUOY_COL:      buoy,
                "Hs__m":       hs,
                "Te__s":       te,
                WPF_COL:       wpf,
                TARGET_COL:    energy,
                EPOCH_COL:     epoch,
            }))

    df_raw: pd.DataFrame = pd.concat(records, ignore_index=True)
    os.makedirs("dataset2", exist_ok=True)
    df_raw.to_csv(DATA_PATH, index=False)
    logger.info("Synthetic CSV saved to: %s", DATA_PATH)
    df = load_and_prepare(DATA_PATH)

# ------------------------------------------------------------------
# Execution Pipeline
# ------------------------------------------------------------------
params: Dict = fit_sfa_epoch1(df)
df = score_efficiency(df, params)
df = compute_generation_deficit(df, params)

rolling: pd.DataFrame = aggregate_rolling(df)
print_degradation_report(df)

epoch_bounds: Dict[int, pd.Timestamp] = _epoch_boundaries(df)

plot_timeseries(rolling, epoch_bounds, save_path=str(PLOT_DIR / "wec_phase2_SFA_sfa_timeseries.png"))
plot_residual_decomposition(df, save_path=str(PLOT_DIR / "wec_phase2_SFA_sfa_residuals.png"))
plot_triple_frontier(df, params, save_path=str(PLOT_DIR / "wec_phase2_sfa_triple_frontier_epoch1.png"), epoch=1)
plot_triple_frontier(df, params, save_path=str(PLOT_DIR / "wec_phase2_sfa_triple_frontier_epoch2.png"), epoch=2)
plot_triple_frontier(df, params, save_path=str(PLOT_DIR / "wec_phase2_sfa_triple_frontier_epoch3.png"), epoch=3)

export_artefacts(df)

logger.info("Phase 2 SFA complete. Output files written to: %s", PLOT_DIR.resolve())
logger.info("=" * 72)

# df.head() # Descomentar para visualizar os dados

2026-07-25 22:46:42 | INFO | ========================================================================
2026-07-25 22:46:42 | INFO | WEC Phase 2 -- Stochastic Frontier Analysis (SFA)
2026-07-25 22:46:42 | INFO | ========================================================================
2026-07-25 22:46:42 | INFO | Loading data from: dataset1/final_dataset/final_wec_fleet_2026.csv
2026-07-25 22:46:42 | INFO | Raw shape: (70272, 22)
2026-07-25 22:46:42 | WARNING | Dropped 4380 rows with non-positive WPF or output
2026-07-25 22:46:42 | INFO | Prepared shape: (65892, 24) | Epochs present: [np.int64(1), np.int64(2), np.int64(3)]
2026-07-25 22:46:42 | INFO | Fitting SFA on Epoch 1 Ramp-up Region: 32880 observations from 12 buoys
2026-07-25 22:46:43 | INFO | MLE results (Epoch 1 frontier):
2026-07-25 22:46:43 | INFO |   beta_0    = +5.060579
2026-07-25 22:46:43 | INFO |   beta_1    = +0.043974  (output elasticity)
2026-07-25 22:46:43 | INFO |   lambda    = 1.238766   (sigma_u / sigma_v)
2026-07-2

### Phase 3 Decision Engine for Wave Energy Converter (WEC) anomaly detection.

#### Methodology
Merges Phase 1 (XGBoost absolute baseline errors) with Phase 2 (Stochastic Frontier Analysis relative efficiency scores) to detect mechanical PTO failures while suppressing environmental false positives caused by atypical sea states.

#### State definitions:
* **0 - Nominal:** A=False, B=False
* **1 - Environmental False Positive:** A=True, B=False
* **2 - Latent Degradation:** A=False, B=True
* **3 - Critical Fault:** A=True, B=True

A 'Maintenance_Alarm' is raised when State 3 density across a 12-hour rolling window (24 half-hour periods) reaches or exceeds the configured threshold.

In [29]:
from __future__ import annotations

import logging
import os
from pathlib import Path
from typing import List

%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ---------------------------------------------------------------------------
# Global constants
# ---------------------------------------------------------------------------

# Input file paths

PATH_PHASE1: str = "dataset1/final_dataset/wec_phase1_outputs.csv"
PATH_PHASE2: str = "dataset1/final_dataset/wec_phase2_outputs.csv"
OUTPUT_DIR: Path = Path("plots/phase3_merge/d1")

# PATH_PHASE1: str = "dataset2/wec_phase1_outputs.csv"
# PATH_PHASE2: str = "dataset2/wec_phase2_outputs.csv"
# OUTPUT_DIR: Path = Path("plots/phase3_merge/d2")



# Merge key columns
MERGE_KEYS: List[str] = ["PCTimeStamp", "Buoy_ID"]

# Condition A threshold multiplier (absolute error)
CONDITION_A_MULTIPLIER: float = -1

# Condition B threshold (relative efficiency floor)
CONDITION_B_EFFICIENCY_THRESHOLD: float = 0.60

# Epochs to focus on for the terminal report and visualisation
REPORT_EPOCHS: List[int] = [1, 2, 3]

# Buoy groupings for the comparative visualisation
HEALTHY_FLEET: List[str] = [f"Boia_{i}" for i in range(1, 9)]
DEGRADED_FLEET: List[str] = [f"Boia_{i}" for i in range(9, 13)]

# State colour palette (0=nominal, 1=env FP, 2=latent, 3=critical)
STATE_COLORS: List[str] = ["#2ecc71", "#f39c12", "#3498db", "#e74c3c"]
STATE_LABELS: List[str] = [
    "State 0 - Nominal",
    "State 1 - Env. False Positive",
    "State 2 - Latent Degradation",
    "State 3 - Critical Fault",
]

# Logging format
LOG_FORMAT: str = "[%(asctime)s] %(levelname)s | %(name)s | %(message)s"
LOG_DATE_FORMAT: str = "%Y-%m-%d %H:%M:%S"

# ---------------------------------------------------------------------------
# Logging setup
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format=LOG_FORMAT,
    datefmt=LOG_DATE_FORMAT,
)
logger: logging.Logger = logging.getLogger("phase3_merge")

#### Core Functions
Definição das funções para carregamento dos artefatos intermédios (Fase 1 e Fase 2), merge de dados, avaliação de condições booleanas, atribuição de estados de operação e geração de matrizes visuais.

In [30]:
# ---------------------------------------------------------------------------
# I/O helpers
# ---------------------------------------------------------------------------
def load_phase1(path: str) -> pd.DataFrame:
    logger.info("Loading Phase 1 data from: %s", path)
    df = pd.read_csv(
        path,
        parse_dates=["PCTimeStamp"],
        dtype={
            "Buoy_ID": str,
            "Predicted_Energy_kW": float,
            "Absolute_Residual": float,
            "Is_Absolute_Anomaly": bool,
            "RMSE_test_dynamic": float,
        },
    )
    logger.info("Phase 1 loaded: %d rows, %d columns.", len(df), df.shape[1])
    return df

def load_phase2(path: str) -> pd.DataFrame:
    logger.info("Loading Phase 2 data from: %s", path)
    df = pd.read_csv(
        path,
        parse_dates=["PCTimeStamp"],
        dtype={
            "Buoy_ID": str,
            "Epoch_Marker": int,
            "SFA_Efficiency": float,
            "Generation_Deficit_kW": float,
        },
    )
    logger.info("Phase 2 loaded: %d rows, %d columns.", len(df), df.shape[1])
    return df

# ---------------------------------------------------------------------------
# Merge
# ---------------------------------------------------------------------------
def merge_phases(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    logger.info("Merging Phase 1 (%d rows) and Phase 2 (%d rows) on %s.", len(df1), len(df2), MERGE_KEYS)
    merged = pd.merge(df1, df2, on=MERGE_KEYS, how="inner")
    logger.info("Merge complete: %d rows retained (inner join).", len(merged))
    return merged

# ---------------------------------------------------------------------------
# Business logic: instantaneous state scoring
# ---------------------------------------------------------------------------
def compute_conditions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Cond_A"] = (df["Absolute_Residual"] < (CONDITION_A_MULTIPLIER * df["RMSE_test_dynamic"]))
    df["Cond_B"] = df["SFA_Efficiency"] < CONDITION_B_EFFICIENCY_THRESHOLD
    logger.debug("Condition A triggered on %d rows; Condition B on %d rows.", df["Cond_A"].sum(), df["Cond_B"].sum())
    return df

def assign_operational_state(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    conditions = [
        (~df["Cond_A"]) & (~df["Cond_B"]),  # State 0
        df["Cond_A"] & (~df["Cond_B"]),      # State 1
        (~df["Cond_A"]) & df["Cond_B"],      # State 2
        df["Cond_A"] & df["Cond_B"],         # State 3
    ]
    df["Operational_State"] = np.select(conditions, [0, 1, 2, 3], default=0)
    state_counts = df["Operational_State"].value_counts().sort_index()
    for state, count in state_counts.items():
        logger.info("Instantaneous State %d count: %d (%.2f%%)", state, count, 100.0 * count / len(df))
    return df

# ---------------------------------------------------------------------------
# Terminal report
# ---------------------------------------------------------------------------
def generate_terminal_report(df: pd.DataFrame) -> None:
    period_duration_hours: float = 0.5
    for epoch in REPORT_EPOCHS:
        epoch_df = df[df["Epoch_Marker"] == epoch].copy()
        if epoch_df.empty:
            logger.warning("No data found for Epoch %d. Terminal report aborted.", epoch)
            continue

        logger.info("=" * 70)
        logger.info("WEC FLEET O&M DISPATCH REPORT  --  EPOCH %d", epoch)
        logger.info("=" * 70)

        is_state3 = epoch_df["Operational_State"] == 3
        state3_summary = (
            epoch_df[is_state3].groupby("Buoy_ID").size()
            .mul(period_duration_hours)
            .rename("State3_Hours")
            .reindex(epoch_df["Buoy_ID"].unique(), fill_value=0.0)
            .reset_index()
            .sort_values("Buoy_ID")
        )

        dispatch_required: List[str] = []

        for _, row in state3_summary.iterrows():
            buoy: str = row["Buoy_ID"]
            state3_hours: float = row["State3_Hours"]
            if state3_hours > 0.0:
                dispatch_required.append(buoy)
                logger.info("  %s | Critical Fault (State 3): %6.1f h | Status: DISPATCH REQUIRED", buoy, state3_hours)
            else:
                logger.info("  %s | Critical Fault (State 3): %6.1f h | Status: Nominal/Latent - no immediate action", buoy, state3_hours)

        logger.info("-" * 70)
        if dispatch_required:
            logger.info("O&M VESSEL DISPATCH REQUIRED FOR: %s", ", ".join(dispatch_required))
        else:
            logger.info("All buoys nominal during Epoch %d. No dispatch required.", epoch)
        logger.info("=" * 70)

# ---------------------------------------------------------------------------
# Visualisation
# ---------------------------------------------------------------------------
def _build_state_pivot(epoch_df: pd.DataFrame, buoy_list: List[str]) -> pd.DataFrame:
    subset = epoch_df[epoch_df["Buoy_ID"].isin(buoy_list)].copy()
    pivot = subset.pivot_table(index="PCTimeStamp", columns="Buoy_ID", values="Operational_State", aggfunc="first")
    ordered_cols = [b for b in buoy_list if b in pivot.columns]
    pivot = pivot[ordered_cols].fillna(0).astype(int)
    
    # Renomear as colunas de Boia_ para Buoy para que o eixo Y do grafico assuma o nome correto
    pivot.columns = [str(col).replace("Boia_", "Buoy ") for col in pivot.columns]
    
    return pivot

def _draw_heatmap_panel(ax: plt.Axes, state_pivot: pd.DataFrame, title: str) -> None:
    cmap = ListedColormap(STATE_COLORS)
    bounds = [-0.5, 0.5, 1.5, 2.5, 3.5]
    norm = BoundaryNorm(bounds, cmap.N)

    im = ax.imshow(state_pivot.T.values, aspect="auto", cmap=cmap, norm=norm, interpolation="nearest", origin="upper")
    n_buoys, n_times = state_pivot.T.shape

    ax.set_yticks(range(n_buoys))
    ax.set_yticklabels(state_pivot.columns.tolist(), fontsize=16)

    timestamps = state_pivot.index
    n_ticks = min(12, n_times)
    tick_step = max(1, n_times // n_ticks)
    tick_positions = list(range(0, n_times, tick_step))
    tick_labels = [timestamps[i].strftime("%m-%d\n%H:%M") for i in tick_positions]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, fontsize=15, rotation=0)

    ax.set_title(title, fontsize=20, fontweight="bold", pad=8)
    ax.set_xlabel("Timestamp (UTC)", fontsize=18)
    ax.set_ylabel("Buoy ID", fontsize=18)
    return im

def generate_visualisation(df: pd.DataFrame) -> None:
    """Produce and save the Phase 3 decision matrix visualisation.

    Creates a two-panel temporal heatmap for Epoch 3 comparing the healthy
    fleet (Buoys 1-8) against the degraded fleet (Buoys 9-12).

    Parameters
    ----------
    df:
        Full merged and annotated DataFrame.
    """
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for epoch in REPORT_EPOCHS:
        epoch_df = df[df["Epoch_Marker"] == epoch].copy()
        if epoch_df.empty:
            logger.warning(
                "No Epoch %d data available; visualisation skipped.", epoch
            )
            continue

        logger.info(
            "Generating visualisation for Epoch %d: %d rows.", epoch, len(epoch_df)
        )

        # Build pivot tables for both fleet groups
        healthy_states = _build_state_pivot(epoch_df, HEALTHY_FLEET)
        degraded_states = _build_state_pivot(epoch_df, DEGRADED_FLEET)

        # Figure layout: 2 rows (healthy / degraded), 1 column
        n_healthy = len(healthy_states.columns)
        n_degraded = len(degraded_states.columns)
        row_h_healthy = max(2.0, n_healthy * 0.55)
        row_h_degraded = max(2.0, n_degraded * 0.55)

        fig, axes = plt.subplots(
            nrows=2,
            ncols=1,
            figsize=(18, row_h_healthy + row_h_degraded + 3),
            gridspec_kw={"height_ratios": [n_healthy, n_degraded]},
            facecolor="white",
        )

        for ax in axes:
            ax.set_facecolor("white")

        _draw_heatmap_panel(
            axes[0],
            healthy_states,
            f"Epoch {epoch} - Healthy Fleet (Buoys 1-8) - Operational State Matrix",
        )
        _draw_heatmap_panel(
            axes[1],
            degraded_states,
            f"Epoch {epoch} - Degraded Fleet (Buoys 9-12) - Operational State Matrix",
        )

        # Shared legend formatted for light background
        legend_elements = [
            Patch(facecolor=c, edgecolor="black", linewidth=1.0, label=lbl)
            for c, lbl in zip(STATE_COLORS, STATE_LABELS)
        ]

        fig.legend(
            handles=legend_elements,
            loc="lower center",
            ncol=4,
            fontsize=16,
            framealpha=0.9,
            facecolor="white",
            edgecolor="black",
            labelcolor="black",
            bbox_to_anchor=(0.5, 0.01),
        )

        # Global title formatting for light background
        fig.suptitle(
            f"WEC Phase 3 Decision Matrix  |  Epoch {epoch}  |  "
            "Healthy vs. Degraded Fleet Comparison",
            fontsize=22,
            fontweight="bold",
            color="black",
            y=0.99,
        )

        # Style axes text and spines for light background
        for ax in axes:
            ax.tick_params(colors="black")
            ax.xaxis.label.set_color("black")
            ax.yaxis.label.set_color("black")
            ax.title.set_color("black")
            for spine in ax.spines.values():
                spine.set_edgecolor("black")
                spine.set_linewidth(1.0)

        plt.tight_layout(rect=[0, 0.06, 1, 0.97])
        
        # Dynamic output path based on the current epoch
        plot_path = OUTPUT_DIR / f"wec_phase3_decision_matrix_epoch_{epoch}.png"
        
        # Export in high resolution suitable for academic publishing
        fig.savefig(plot_path, dpi=600, bbox_inches="tight", facecolor=fig.get_facecolor())
        plt.close(fig)
        logger.info("Visualisation saved to: %s", plot_path)

#### Execution Block
Chamada sequencial das funções para correr a Fase 3 na totalidade e avaliar a performance global do sistema (substitui a função `run_pipeline`).

In [31]:
logger.info("=" * 70)
logger.info("WEC Phase 3 -- Decision Engine Initialised")
logger.info("=" * 70)

# --- Step 1: Load inputs ---
df_phase1 = load_phase1(PATH_PHASE1)
df_phase2 = load_phase2(PATH_PHASE2)

# --- Step 2: Merge ---
df_merged = merge_phases(df_phase1, df_phase2)

# --- Step 3: Diagnostic conditions ---
df_conditions = compute_conditions(df_merged)

# --- Step 4: Operational state assignment ---
df_states = assign_operational_state(df_conditions)

# --- Step 5: Terminal report ---
generate_terminal_report(df_states)

# --- Step 6: Visualisation ---
generate_visualisation(df_states)

logger.info("Phase 3 pipeline complete.")



2026-07-25 22:46:57 | INFO | ======================================================================
2026-07-25 22:46:57 | INFO | WEC Phase 3 -- Decision Engine Initialised
2026-07-25 22:46:57 | INFO | ======================================================================
2026-07-25 22:46:57 | INFO | Loading Phase 1 data from: dataset1/final_dataset/wec_phase1_outputs.csv
2026-07-25 22:46:57 | INFO | Phase 1 loaded: 66120 rows, 6 columns.
2026-07-25 22:46:57 | INFO | Loading Phase 2 data from: dataset1/final_dataset/wec_phase2_outputs.csv
2026-07-25 22:46:57 | INFO | Phase 2 loaded: 65892 rows, 5 columns.
2026-07-25 22:46:57 | INFO | Merging Phase 1 (66120 rows) and Phase 2 (65892 rows) on ['PCTimeStamp', 'Buoy_ID'].
2026-07-25 22:46:57 | INFO | Merge complete: 65892 rows retained (inner join).
2026-07-25 22:46:57 | INFO | Instantaneous State 0 count: 33605 (51.00%)
2026-07-25 22:46:57 | INFO | Instantaneous State 1 count: 27643 (41.95%)
2026-07-25 22:46:57 | INFO | Instantaneous State 